# 14 · The router: one entry point, A2A routing

Stage 3, step 4. Until now **you** picked the agent: `okf.send(...)` or `scout.send(...)`. The **router** is a third agent that does it for you. You talk only to the router; it reads the other agents' cards, decides who should answer, and forwards the message over A2A. Later the Chainlit UI will talk only to the router.

```
                                   ┌──► A2A ──► okf_agent     "What did GPT-4 score on the bar exam?"
 you ──► okf_router ── route ──────┼──► A2A ──► paper_scout   "Find recent papers on MoE"
          (A2A agent too)          ├──► both, in parallel      "What does the bundle say about RLHF, and newer papers?"
                                   └──► direct reply           "hi", "write me a poem"
```

The router is itself an A2A agent (it has a card), and it **calls** other A2A agents. That's the "agent-to-agent" in A2A.

Code: `okf_agent/router.py`, plus a third entry in `langgraph.json`.

## 0. Restart the backend

`start_backend.bat` changed: the agent server now starts with `--n-jobs-per-worker 4` (section 6 explains why: without it the router **deadlocks**). Close the three windows, run `start_backend.bat` again, wait for all three to be ready.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import socket
import time

import pandas as pd
import psycopg

from okf_agent import AgentConfig
from okf_agent.a2a_client import A2AClient, langgraph_card_url

AGENT_URL = os.getenv("OKF_AGENT_URL", "http://127.0.0.1:2024")
config = AgentConfig.load()

def port_open(url):
    host, port = url.split("//")[1].split("/")[0].split(":")
    with socket.socket() as s:
        s.settimeout(2)
        return s.connect_ex((host, int(port))) == 0

for name, url in [("OKF MCP server", config.mcp_url), ("arXiv MCP server", config.arxiv_mcp_url), ("agent server", AGENT_URL)]:
    print(f"{name:17s} up: {port_open(url)}")

## 1. The router knows the agents only by their cards

`AgentDirectory` reads the A2A card of each agent named in `OKF_ROUTER_AGENTS` (default `okf_agent,paper_scout`). The **descriptions** on those cards (written in `langgraph.json`) are all the router knows about what each agent does. That's why we wrote them for a reader that has never seen the project.

In [ ]:
from okf_agent.router import AgentDirectory

directory = AgentDirectory(AGENT_URL, ["okf_agent", "paper_scout"])
print(directory.describe())

## 2. How `route` decides

```python
if GREETING_RE.match(question):                      # 1. "hi", "thanks", "what can you do"
    chosen, by = [], "rule-greeting"                 #    → direct reply, no model call
else:                                                # 2. qwen decides, from the cards
    reply = llm.complete(ROUTE_PROMPT.format(agents=directory.describe()), message)
    parsed = parse_route(reply.text, allowed)        #    {"agents": [...]} → only known names
    if parsed is not None:
        chosen, by = parsed, "llm"
    elif last and looks_like_followup(question):     # 3. qwen's reply unusable: rules
        chosen, by = last, "rule-followup"
    else:
        chosen, by = keyword_route(question, allowed), "rule-keywords"
```

- **qwen sees the previous message too**: `Previous message (answered by okf_agent): What score did GPT-4 get…? / New message: And GPT-3.5?`. So it can tell a follow-up ("And GPT-3.5?" → same agent) from a new request.
- A word rule can't do that. My first version reused the last agent for any short message or any message containing "it". Then "write me a poem" went to the paper scout, and "…newer papers on **it**?" counted as a follow-up.
- **`parse_route` only accepts names that exist.** If qwen invents an agent, the name is dropped. If nothing usable is left, the rules decide. qwen can't send a message anywhere we didn't allow.

These helpers are plain functions, so you can try them without the server:

In [ ]:
from okf_agent.router import keyword_route, parse_route, GREETING_RE

allowed = ["okf_agent", "paper_scout"]
for text in ['{"agents": ["paper_scout"]}', '```json\n{"agents": ["okf_agent", "paper_scout"]}\n```',
             '{"agents": []}', '{"agents": ["web_search"]}', "I think the okf agent"]:
    print(f"{text!r:60} → {parse_route(text, allowed)}")
print()
for q in ["Any new papers on RLHF?", "What is the GPT-4 context length?"]:
    print(f"keyword rule: {q!r:40} → {keyword_route(q, allowed)}")
print()
for q in ["hi!", "what can you do?", "hi, what score did GPT-4 get?"]:
    print(f"greeting rule: {q!r:35} → {bool(GREETING_RE.match(q))}")

## 3. The graph: parallel branches

```
                       ┌─► ask_okf_agent ───┐
    START → route ─────┼─► ask_paper_scout ─┼─► combine → log → END
                       └─► direct ──────────┘
```

```python
def after_route(state):
    return [f"ask_{a}" for a in state["route"]] or ["direct"]     # a LIST of next nodes

g.add_conditional_edges("route", after_route, ask_nodes + ["direct"])
for node in ask_nodes + ["direct"]:
    g.add_edge(node, "combine")
```

New compared to step 13: the conditional edge returns a **list**. One name → one branch. **Two names → both nodes run at the same time** (LangGraph calls this a *superstep*). `combine` runs once, after every branch that ran has finished.

Two branches writing to the same state key at the same time need a rule for merging, so `replies` and `contexts` get a **reducer**, just like `messages` has `add_messages`:

```python
def merge_dict(old, new):
    if new is None:                  # route sends None at the start of each turn = reset
        return {}
    return {**(old or {}), **new}    # ask_okf_agent adds {"okf_agent": …}, ask_paper_scout adds {"paper_scout": …}

class RouterState(AgentState, total=False):
    replies:  Annotated[dict, merge_dict]    # this turn's answers
    contexts: Annotated[dict, merge_dict]    # {agent: A2A contextId}
```

Without the reducer, LangGraph would raise an error: two nodes updated the same key in one step and it can't know which wins.

In [ ]:
from okf_agent.router import build_router_graph
print(build_router_graph(AgentDirectory(AGENT_URL, ["okf_agent", "paper_scout"]), None).compile().get_graph().draw_mermaid())

## 4. One A2A conversation per agent, only for real follow-ups

The router has **its own** conversation with you (its `contextId`). Behind it, it opens conversations with the agents:

```python
context_id = state.get("contexts", {}).get(agent) if state["followup"] else None
r = client.send(state["question"], context_id=context_id)
```

- **Follow-up** (same agent as last turn): reuse that agent's conversation, so the agent's own memory makes "And GPT-3.5?" work.
- **New request**: a fresh conversation, so the agent can't mistake it for a follow-up of something older. (Before this, "…newer papers on **it**?" got glued to "And GPT-3.5?" in `okf_agent`'s search.)

## 5. Talk to the router

`chat()` sends a message to the router over A2A, then reads the router's `qa_log` row to show **where the message went and who decided**.

In [ ]:
router = A2AClient.from_card(langgraph_card_url(AGENT_URL, "okf_router"))
print(router.name, "→", router.url, "\n")
conversation = {"id": None}

def chat(message, show=600):
    t0 = time.perf_counter()
    r = router.send(message, context_id=conversation["id"])
    conversation["id"] = r.context_id
    with psycopg.connect(config.postgres_url) as conn:
        row = conn.execute("SELECT search_query, followup FROM qa_log WHERE agent = 'okf_router' AND thread_id = %s "
                           "ORDER BY id DESC LIMIT 1", (r.context_id,)).fetchone()
    route, followup = row if row else ("?", None)
    print(f"YOU: {message}\nROUTE {route}{'  [follow-up]' if followup else ''}  ·  {r.state}  ·  {time.perf_counter() - t0:.1f} s")
    print("-" * 100)
    print(r.text[:show] + (" …" if len(r.text) > show else ""), "\n")

chat("hi")

In [ ]:
chat("What score did GPT-4 get on the bar exam?")
chat("And GPT-3.5?")

In [ ]:
chat("Find recent papers on mixture of experts language models")
chat("only ones since 2025")

In [ ]:
chat("What does the bundle say about RLHF, and are there newer papers on it?", show=2500)

In [ ]:
chat("write me a poem about cats")

What to look at:
- **ROUTE** for each message: `direct (rule-greeting)`, `okf_agent (llm)`, `paper_scout (llm)`, `okf_agent, paper_scout (llm)`.
- The two follow-ups should say `[follow-up]` and stay with the same agent.
- The RLHF question: **two headings**, "From the bundle" and "On arXiv". The two agents ran in parallel, but your single Ollama answers one request at a time, so the total is roughly the sum of both.
- Time: the router's own decision is one small qwen call (about 1 s) on top of the agent's time.

## 6. The deadlock (why `--n-jobs-per-worker 4`)

`langgraph dev` runs graphs as **jobs**, and by default it runs **one job at a time**. The router's run *is* a job, and while it waits for `okf_agent`'s answer it keeps that one slot. `okf_agent`'s run is queued behind it, waiting for the slot the router will never free. Neither moves: a **deadlock**. In my test the router hung until its 60 s timeout.

`--n-jobs-per-worker 4` gives the server 4 slots: the router plus both agents at once, with one spare. This is a classic problem whenever a service calls itself. In production each agent would usually be its own deployment, and then it can't happen.

## 7. The log now shows the whole chain

Every router turn writes a row (`agent = okf_router`, `search_query` = the route), and each agent it called writes its own row. The router's `thread_id` is your conversation; the agents' `thread_id`s are the router's conversations with them.

In [ ]:
with psycopg.connect(config.postgres_url) as conn:
    cur = conn.execute("""SELECT id, agent, followup, left(question, 60) AS question, search_query, refused,
                                 round(total_ms) AS ms, left(thread_id, 8) AS thread
                          FROM qa_log ORDER BY id DESC LIMIT 14""")
    log = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
log

## Where we are

- ✅ **10** okf_mcp · ✅ **11** okf_agent · ✅ **12** A2A service · ✅ **13** paper_scout
- ✅ **14** **okf_router**: cards → qwen picks with the previous message in view (rules as fallback) → A2A to one agent, both in parallel, or a direct reply; fresh agent conversations for new requests
- ⏭ **15** report agent: turns a conversation's answers into a Markdown report file, through the official **filesystem MCP server** (limited to a `reports/` folder). Needs **Node.js** (`node --version`).
- then **stage 4**: guardrails as graph nodes (input checks before `route`; output checks using `bad_citations`)